# Retention Analysis: The King of Metrics

Retention is often called the "king of metrics" in product analytics because it directly measures whether users find ongoing value in your product. Unlike acquisition (a one-time event), retention reveals the true strength of your product-market fit.

## The Leaky Bucket Analogy

Think of your user base as a bucket of water:
- **Acquisition** adds water to the bucket
- **Retention** prevents water from leaking out
- **Growth** = (acquisition rate) - (leakage rate)

Even with fantastic acquisition, a leaky bucket won't grow. That's why retention analysis is critical.

## What We'll Cover

This notebook explores **5 complementary methods** for analyzing retention:

1. **N-Week Retention Curves** — How many users are active N weeks after signup?
2. **Rolling vs Classic Retention** — How do definitions affect the story?
3. **Kaplan-Meier Survival Analysis** — When do users churn?
4. **Curve Fitting** — Can we predict future retention?
5. **Churn Risk Scoring** — Which early behaviors predict churn?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Color palette
COLORS = {
    'primary': '#2E86AB',
    'secondary': '#F18F01',
    'accent1': '#2CA58D',
    'accent2': '#E15554'
}

print('Libraries loaded successfully!')

In [ ]:
# Load data
df = pd.read_csv('../data/inputs/activity_clean.csv')

print(f'Shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nFirst 5 rows:')
print(df.head())
print(f'\nData types:')
print(df.dtypes)
print(f'\nMissing values:')
print(df.isnull().sum())

## Data Preparation

We'll compute weeks since signup for each user, which is the foundation for all retention metrics.

In [ ]:
# Convert dates to datetime
df['signup_date'] = pd.to_datetime(df['signup_date'])
df['activity_date'] = pd.to_datetime(df['activity_date'])

# Compute weeks since signup
df['weeks_since_signup'] = (df['activity_date'] - df['signup_date']).dt.days // 7

# Only keep activity within 12 weeks of signup
df_retention = df[df['weeks_since_signup'] >= 0].copy()
df_retention = df_retention[df_retention['weeks_since_signup'] <= 11].copy()

print(f'Activity records (0-11 weeks): {len(df_retention)}')
print(f'Unique users: {df_retention["user_id"].nunique()}')
print(f'Date range: {df_retention["signup_date"].min()} to {df_retention["activity_date"].max()}')

In [ ]:
# Build user-level summary: for each user, which weeks were they active?
# Also capture early behavior features
user_summary = df_retention.groupby('user_id').agg({
    'signup_date': 'first',
    'signup_cohort': 'first',
    'plan_type': 'first',
    'weeks_since_signup': lambda x: set(x),  # Set of weeks active
    'session_duration_sec': ['mean', 'sum'],
    'event_type': 'nunique',  # Diversity of event types
    'activity_date': 'count'  # Total activities
}).reset_index()

user_summary.columns = ['user_id', 'signup_date', 'signup_cohort', 'plan_type', 
                         'active_weeks_set', 'avg_session_duration', 'total_session_duration',
                         'event_diversity', 'total_activities']

print(f'User summary shape: {user_summary.shape}')
print(f'\nSample:')
print(user_summary.head())

In [ ]:
# Create binary indicator: active in each week (0-11)
for week in range(12):
    user_summary[f'active_week_{week}'] = user_summary['active_weeks_set'].apply(lambda x: 1 if week in x else 0)

# Define churn_week: the last week a user was active
user_summary['churn_week'] = user_summary['active_weeks_set'].apply(
    lambda x: max(x) if len(x) > 0 else -1
)

print('Active week flags and churn_week computed.')
print(f'\nChurn week distribution:')
print(user_summary['churn_week'].value_counts().sort_index())

## Method 1: N-Week Retention Curves

**Definition:** The percentage of users active in week N (where week 0 is signup week).

**What it shows:** How many users return in each week post-signup.

**Calculation:**
- Week 0: % of all users active in their first week
- Week 1: % of all users active 1 week after signup
- ... Week 11: % of all users active 11 weeks after signup

This is the **most intuitive retention metric** and the standard used in most companies.

In [ ]:
# Compute N-week retention (as % of total users)
retention_by_week = []
for week in range(12):
    pct_active = (user_summary[f'active_week_{week}'].sum() / len(user_summary)) * 100
    retention_by_week.append(pct_active)

retention_df = pd.DataFrame({
    'week': range(12),
    'pct_retained': retention_by_week
})

print('N-Week Retention Curve:')
print(retention_df.to_string(index=False))
print(f'\nKey insight: {retention_df.iloc[0, 1]:.1f}% active at signup, {retention_df.iloc[1, 1]:.1f}% in week 1.')
print(f'Week-over-week drop from week 0→1: {retention_df.iloc[0, 1] - retention_df.iloc[1, 1]:.1f}pp')

In [ ]:
# Plot N-week retention curve
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(retention_df['week'], retention_df['pct_retained'], 
        marker='o', linewidth=2.5, markersize=8, color=COLORS['primary'], label='Retention %')

# Highlight key weeks
key_weeks = [1, 4, 8, 11]
for week in key_weeks:
    if week < len(retention_df):
        ax.axvline(x=week, alpha=0.2, linestyle='--', color='gray')
        ax.text(week, retention_df.iloc[week, 1] + 2, f'W{week}', ha='center', fontsize=9, color='gray')

ax.set_xlabel('Weeks Since Signup', fontsize=12, fontweight='bold')
ax.set_ylabel('% Users Active', fontsize=12, fontweight='bold')
ax.set_title('N-Week Retention Curve', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 105)
ax.set_xticks(range(12))

plt.tight_layout()
plt.savefig('../data/outputs/nb05/nb05_01_nweek_retention_curve.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: nb05_01_nweek_retention_curve.png')

### Benefits & Limitations

**Benefits:**
- ✓ Intuitive and easy to communicate to stakeholders
- ✓ Directly answers "how many users stay?"
- ✓ Enables cohort comparison
- ✓ Industry standard

**Limitations:**
- ✗ Doesn't distinguish between "active once" and "consistently active"
- ✗ Doesn't account for user who churned in week 3 but reactivated in week 8
- ✗ Assumes uniform definition of "active" across all users

## Method 2: Rolling vs Classic Retention

These two definitions can tell very different stories:

- **Classic Retention:** % of users active in week N *and only* week N (cohort-specific)
- **Rolling Retention:** % of users active in week N *or any week after* (cumulative)

The difference matters for understanding user behavior patterns.

In [ ]:
# Method 2: Classic vs Rolling Retention
# Classic: active specifically in week N
# Rolling: active in week N or any subsequent week

classic_retention = []
rolling_retention = []

for week in range(12):
    # Classic: active in this week specifically
    classic = (user_summary[f'active_week_{week}'].sum() / len(user_summary)) * 100
    classic_retention.append(classic)
    
    # Rolling: still active at or after this week (max active week >= current week)
    rolling = (user_summary['churn_week'].apply(lambda x: x >= week).sum() / len(user_summary)) * 100
    rolling_retention.append(rolling)

retention_comparison = pd.DataFrame({
    'week': range(12),
    'classic': classic_retention,
    'rolling': rolling_retention
})

print('Classic vs Rolling Retention:')
print(retention_comparison.to_string(index=False))

In [ ]:
# Plot both
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(retention_comparison['week'], retention_comparison['classic'], 
        marker='o', linewidth=2.5, markersize=8, color=COLORS['primary'], label='Classic (Week N only)')
ax.plot(retention_comparison['week'], retention_comparison['rolling'], 
        marker='s', linewidth=2.5, markersize=8, color=COLORS['secondary'], label='Rolling (Week N or later)')

ax.set_xlabel('Weeks Since Signup', fontsize=12, fontweight='bold')
ax.set_ylabel('% Users Retained', fontsize=12, fontweight='bold')
ax.set_title('Classic vs Rolling Retention', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 105)
ax.set_xticks(range(12))

plt.tight_layout()
plt.savefig('../data/outputs/nb05/nb05_02_classic_vs_rolling.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: nb05_02_classic_vs_rolling.png')

### Benefits & Limitations

**Classic Retention:**
- ✓ Shows sustained engagement over time
- ✗ Counts churn and reactivation separately

**Rolling Retention:**
- ✓ Accounts for lifetime value early
- ✗ Hides whether users are continuously active or sporadic

## Method 3: Kaplan-Meier Survival Analysis

**What it does:** Models the probability that a user "survives" (remains active) beyond week N, accounting for right-censoring (users we can't fully observe).

**Key concept:** We observe each user until they churn. Once they churn, they're out. This is classic survival analysis.

**Advantage:** Handles the reality that not all users are observed for the full 12 weeks (some may have joined recently).

In [ ]:
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# Prepare data for Kaplan-Meier
# Duration: churn_week (when user became inactive)
# Event: 1 if we observed churn, 0 if still active (censored)
user_summary['duration'] = user_summary['churn_week'] + 1  # Convert to actual weeks
user_summary['event_observed'] = (user_summary['churn_week'] < 11).astype(int)  # Churn before end of observation

print('Kaplan-Meier preparation:')
print(f'  Users with observed churn: {user_summary["event_observed"].sum()}')
print(f'  Censored users (still active at week 11): {(1 - user_summary["event_observed"]).sum()}')
print(f'\nDuration range: {user_summary["duration"].min()} to {user_summary["duration"].max()}')

In [ ]:
# Fit Kaplan-Meier
kmf = KaplanMeierFitter()
kmf.fit(durations=user_summary['duration'], event_observed=user_summary['event_observed'], label='Overall')

print('Kaplan-Meier Survival Curve:')
print(kmf.survival_function_)
print(f'\nMedian survival time (weeks): {kmf.median_survival_time_:.2f}')

In [ ]:
# Plot Kaplan-Meier curve
fig, ax = plt.subplots(figsize=(12, 6))

kmf.plot_survival_function(ax=ax, ci_show=True, color=COLORS['primary'], linewidth=2.5)

ax.set_xlabel('Weeks Since Signup', fontsize=12, fontweight='bold')
ax.set_ylabel('Probability of Survival (Retention)', fontsize=12, fontweight='bold')
ax.set_title('Kaplan-Meier Survival Curve', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('../data/outputs/nb05/nb05_03_kaplan_meier.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: nb05_03_kaplan_meier.png')

In [ ]:
# Compare survival by plan_type
fig, ax = plt.subplots(figsize=(12, 6))

for plan in user_summary['plan_type'].unique():
    mask = user_summary['plan_type'] == plan
    kmf_plan = KaplanMeierFitter()
    kmf_plan.fit(durations=user_summary[mask]['duration'], 
                 event_observed=user_summary[mask]['event_observed'],
                 label=f'{plan} (n={mask.sum()})')
    kmf_plan.plot_survival_function(ax=ax, ci_show=False, linewidth=2.5)

ax.set_xlabel('Weeks Since Signup', fontsize=12, fontweight='bold')
ax.set_ylabel('Probability of Survival', fontsize=12, fontweight='bold')
ax.set_title('Kaplan-Meier Survival by Plan Type', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('../data/outputs/nb05/nb05_03_km_by_plan.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: nb05_03_km_by_plan.png')

In [ ]:
# Log-rank test: are survival curves different by plan?
plans = user_summary['plan_type'].unique()
if len(plans) >= 2:
    plan_a, plan_b = plans[0], plans[1]
    mask_a = user_summary['plan_type'] == plan_a
    mask_b = user_summary['plan_type'] == plan_b
    
    results = logrank_test(
        durations_A=user_summary[mask_a]['duration'],
        durations_B=user_summary[mask_b]['duration'],
        event_observed_A=user_summary[mask_a]['event_observed'],
        event_observed_B=user_summary[mask_b]['event_observed']
    )
    
    print(f'\nLog-Rank Test: {plan_a} vs {plan_b}')
    print(f'  Test statistic: {results.test_statistic:.4f}')
    print(f'  p-value: {results.p_value:.4f}')
    print(f'  Significant at α=0.05: {"Yes" if results.p_value < 0.05 else "No"}')

### Benefits & Limitations

**Benefits:**
- ✓ Accounts for right-censoring (users observed for partial duration)
- ✓ Powerful statistical test (log-rank) for comparing groups
- ✓ Well-established methodology from medical research
- ✓ Provides median survival time

**Limitations:**
- ✗ Assumes churn is absorbing (users don't reactivate)
- ✗ Requires intact lifelines library
- ✗ Less intuitive for non-technical stakeholders

## Method 4: Curve Fitting

**Idea:** Retention often follows predictable mathematical patterns. We can fit exponential or power-law curves to estimate future retention.

- **Exponential:** R(t) = a × exp(-b × t)
- **Power Law:** R(t) = a × t^(-b)

Which fits better? R² tells us.

In [ ]:
from scipy.optimize import curve_fit
from scipy.stats import linregress

# Prepare data: weeks vs retention %
X = retention_df['week'].values.astype(float)
Y = retention_df['pct_retained'].values / 100  # Convert to proportion

# Remove week 0 for fitting (always 1.0, creates fitting issues)
X_fit = X[1:]
Y_fit = Y[1:]

print(f'Fitting data: weeks {X_fit} with retention {Y_fit}')

In [ ]:
# Define model functions
def exponential(t, a, b):
    return a * np.exp(-b * t)

def power_law(t, a, b):
    return a * (t ** (-b))

# Fit exponential
try:
    popt_exp, _ = curve_fit(exponential, X_fit, Y_fit, p0=[1, 0.3], maxfev=5000)
    Y_pred_exp = exponential(X_fit, *popt_exp)
    ss_res_exp = np.sum((Y_fit - Y_pred_exp) ** 2)
    ss_tot = np.sum((Y_fit - Y_fit.mean()) ** 2)
    r2_exp = 1 - (ss_res_exp / ss_tot)
    print(f'Exponential: a={popt_exp[0]:.4f}, b={popt_exp[1]:.4f}, R²={r2_exp:.4f}')
except:
    r2_exp = -1
    popt_exp = [np.nan, np.nan]
    print('Exponential fit failed')

# Fit power law
try:
    popt_pow, _ = curve_fit(power_law, X_fit, Y_fit, p0=[1, 0.5], maxfev=5000)
    Y_pred_pow = power_law(X_fit, *popt_pow)
    ss_res_pow = np.sum((Y_fit - Y_pred_pow) ** 2)
    r2_pow = 1 - (ss_res_pow / ss_tot)
    print(f'Power Law: a={popt_pow[0]:.4f}, b={popt_pow[1]:.4f}, R²={r2_pow:.4f}')
except:
    r2_pow = -1
    popt_pow = [np.nan, np.nan]
    print('Power law fit failed')

print(f'\nBetter fit: {"Exponential" if r2_exp > r2_pow else "Power Law"}')

In [ ]:
# Plot actual vs fitted curves
fig, ax = plt.subplots(figsize=(12, 6))

# Plot actual retention (all weeks)
ax.plot(X, Y * 100, marker='o', linewidth=2.5, markersize=8, 
        color=COLORS['primary'], label='Actual', zorder=3)

# Plot fitted curves
X_smooth = np.linspace(1, 11, 100)
if r2_exp > -0.5:
    Y_exp_smooth = exponential(X_smooth, *popt_exp) * 100
    ax.plot(X_smooth, Y_exp_smooth, linewidth=2, linestyle='--', 
            color=COLORS['secondary'], label=f'Exponential (R²={r2_exp:.3f})')

if r2_pow > -0.5:
    Y_pow_smooth = power_law(X_smooth, *popt_pow) * 100
    ax.plot(X_smooth, Y_pow_smooth, linewidth=2, linestyle='--', 
            color=COLORS['accent1'], label=f'Power Law (R²={r2_pow:.3f})')

ax.set_xlabel('Weeks Since Signup', fontsize=12, fontweight='bold')
ax.set_ylabel('% Users Retained', fontsize=12, fontweight='bold')
ax.set_title('Curve Fitting: Exponential vs Power Law', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 105)
ax.set_xticks(range(12))

plt.tight_layout()
plt.savefig('../data/outputs/nb05/nb05_04_curve_fitting.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: nb05_04_curve_fitting.png')

### Benefits & Limitations

**Benefits:**
- ✓ Enables prediction of future retention
- ✓ Reveals whether retention follows known mathematical patterns
- ✓ Useful for forecasting and capacity planning

**Limitations:**
- ✗ Past patterns don't guarantee future behavior
- ✗ Poor fits (low R²) suggest retention isn't following these models
- ✗ Extrapolation beyond observation window is risky

## Method 5: Churn Risk Scoring

**Goal:** Predict which users will churn based on early behavior.

**Features from first week:**
- Sessions in week 1
- Average session duration
- Diversity of event types
- Plan type

**Target:** Churned by week 4 (0=retained, 1=churned)

We'll use logistic regression and inspect feature importance.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

# Compute week-1 specific features
df_w1 = df_retention[df_retention['weeks_since_signup'] == 0].groupby('user_id').agg({
    'session_duration_sec': ['count', 'mean'],
    'event_type': 'nunique'
}).reset_index()

df_w1.columns = ['user_id', 'week1_sessions', 'week1_avg_duration', 'week1_event_diversity']

print('Week 1 features:')
print(df_w1.head())
print(f'\nShape: {df_w1.shape}')
print(f'Missing: {df_w1.isnull().sum().sum()}')

In [ ]:
# Merge with user summary
model_data = user_summary.merge(df_w1, on='user_id', how='left')

# Fill missing week-1 features with 0 (user had no activity in week 1)
model_data[['week1_sessions', 'week1_avg_duration', 'week1_event_diversity']] = \
    model_data[['week1_sessions', 'week1_avg_duration', 'week1_event_diversity']].fillna(0)

# Target: churned by week 4
model_data['churned_by_w4'] = (model_data['churn_week'] < 4).astype(int)

print(f'Churned by week 4: {model_data["churned_by_w4"].sum()} / {len(model_data)}')
print(f'Retention rate to week 4: {(1 - model_data["churned_by_w4"].mean()) * 100:.1f}%')

In [ ]:
# Prepare features
feature_cols = ['week1_sessions', 'week1_avg_duration', 'week1_event_diversity']

# Add plan type dummy variables
plan_dummies = pd.get_dummies(model_data['plan_type'], prefix='plan')
model_data = pd.concat([model_data, plan_dummies], axis=1)
plan_cols = [col for col in plan_dummies.columns]

feature_cols.extend(plan_cols)
X = model_data[feature_cols].copy()
y = model_data['churned_by_w4'].copy()

print(f'Features: {feature_cols}')
print(f'\nFeature correlations with churn:')
for col in feature_cols:
    corr = model_data[col].corr(y)
    print(f'  {col}: {corr:.3f}')

In [ ]:
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit logistic regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_scaled, y)

# Evaluate
y_pred_proba = lr.predict_proba(X_scaled)[:, 1]
y_pred = lr.predict(X_scaled)

auc = roc_auc_score(y, y_pred_proba)
print(f'Model Performance:')
print(f'  ROC-AUC: {auc:.4f}')
print(f'  Accuracy: {(y_pred == y).mean():.4f}')
print(f'\nConfusion Matrix:')
print(confusion_matrix(y, y_pred))
print(f'\nClassification Report:')
print(classification_report(y, y_pred, target_names=['Retained', 'Churned']))

In [ ]:
# Feature importance (coefficients)
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lr.coef_[0],
    'abs_coefficient': np.abs(lr.coef_[0])
}).sort_values('abs_coefficient', ascending=False)

print('Feature Importance (Logistic Regression Coefficients):')
print(feature_importance.to_string(index=False))
print(f'\nIntercept: {lr.intercept_[0]:.4f}')

In [ ]:
# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 6))

colors = [COLORS['accent2'] if x < 0 else COLORS['primary'] for x in feature_importance['coefficient']]
ax.barh(range(len(feature_importance)), feature_importance['coefficient'], color=colors)
ax.set_yticks(range(len(feature_importance)))
ax.set_yticklabels(feature_importance['feature'])
ax.set_xlabel('Coefficient (Standardized)', fontsize=12, fontweight='bold')
ax.set_title('Churn Risk: Feature Importance\n(Red = increases churn risk, Blue = decreases churn risk)', 
             fontsize=13, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../data/outputs/nb05/nb05_05_churn_risk_features.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: nb05_05_churn_risk_features.png')

### Benefits & Limitations

**Benefits:**
- ✓ Actionable: identify high-risk users early
- ✓ Data-driven targeting for retention campaigns
- ✓ Shows which behaviors predict churn
- ✓ Can be deployed as real-time scoring

**Limitations:**
- ✗ Requires labeled historical data
- ✗ Assumes future resembles past
- ✗ Causation unclear (does behavior cause churn or indicate unmotivated users?)
- ✗ Ethical concerns: targeting based on predicted behavior

## Method Comparison Table

| Method | Best For | Difficulty | Interpretability | Predictive |
|--------|----------|-----------|-----------------|------------|
| **N-Week Retention** | Reporting, stakeholder communication | Easy | Very High | No |
| **Rolling vs Classic** | Understanding activity patterns | Easy | High | No |
| **Kaplan-Meier** | Statistical rigor, group comparison | Medium | Medium | No |
| **Curve Fitting** | Forecasting future retention | Medium | Medium | Yes |
| **Churn Risk Scoring** | Identifying at-risk users | Hard | Low | Yes |

**Recommendation:** Use **N-Week Retention + Kaplan-Meier** for analysis. Add **Churn Risk Scoring** for operational use (retention campaigns).

## Key Findings

1. **Retention Cliff:** Most products show dramatic drop-off in weeks 1-2. Our data follows this pattern.

2. **Classic vs Rolling:** The gap between classic and rolling retention reveals how many users churn and reactivate.

3. **Plan Type Matters:** Kaplan-Meier curves by plan type show whether certain segments survive longer. Use log-rank test for significance.

4. **Predictive Power:** Week-1 behavior (sessions, event diversity) is surprisingly predictive of 4-week retention. Early engagement is a strong signal.

5. **Curve Fitting Utility:** If R² > 0.9, exponential/power-law models can forecast. Otherwise, retention is irregular and unpredictable.

---

**Next Steps:**
- Segment by cohort (signup_cohort W01-W12) to detect seasonal trends
- Compare retention before/after product changes
- Build automated dashboards around N-week retention
- Deploy churn risk model to prioritize retention efforts

## Guardrails & North Star in Retention Analysis

---

### Retention IS the North Star (Or Its Closest Proxy)

In many products, retention is the North Star metric itself — or the strongest leading indicator of it. Here's why:

- **Retention × Acquisition = Growth.** If retention is low, no amount of acquisition spending will save the product.
- **Retention predicts revenue.** Retained users generate lifetime value; churned users generate zero.
- **Retention reflects product-market fit.** If people keep coming back, the product is delivering real value.

For our dataset, we can define the North Star as **Week-4 Retention Rate** (the percentage of users still active 4 weeks after signup). Week 4 is a common benchmark because it filters out "tourists" who try once and leave, while being early enough to act on.

**In a SmarterDx context:** the North Star might be "Percentage of hospitals still actively using AI-assisted chart review 90 days post-go-live." If that number is high, the product is working. If it drops, something in the onboarding or product experience is broken.

---

### Guardrail Metrics When Optimizing Retention

When you try to improve retention (e.g., through better onboarding, push notifications, or feature changes), watch these guardrails:

| Retention Optimization | Guardrail | Why It Matters |
|---|---|---|
| More onboarding emails/nudges | Unsubscribe rate, spam complaints | Aggressive messaging can annoy users into leaving faster |
| Gamification / streaks | Session quality (actions per session) | Users may open the app to maintain a streak without actually engaging |
| Reducing friction to activate | Error rate, support tickets | Skipping important setup steps causes problems later |
| Re-engagement campaigns (win-back) | Cost per reactivated user | Diminishing returns on win-back spend |
| Feature gating by plan tier | Upgrade rate AND churn rate by tier | Gating too aggressively pushes free users away instead of converting them |

---

### Churn Guardrails: Automated Alerting

For a real product, you'd set automated alerts for:
- **Week-1 retention drops below X%** — something in onboarding is broken (investigate immediately)
- **Churn rate for any cohort exceeds 2σ above historical mean** — flag for investigation
- **Pro-tier churn exceeds free-tier churn** — this should never happen; it signals a product quality issue with the paid experience
- **Median time-to-churn decreases** — users are leaving faster than before, which is a leading indicator of a bigger problem

---

### What Our Data Shows vs. What You'd Add in Practice

From our analysis, we can calculate retention curves, identify churn timing, and score at-risk users. What we **can't** measure but **should mention in an interview**:
- User satisfaction at each retention milestone (NPS at week 1, 4, 8)
- Reason for churn (survey data or exit interviews)
- Product usage depth (are retained users actually getting value, or just logging in?)
- Revenue retention (are retained users maintaining or increasing their spend?)

**The takeaway:** Retention analysis tells you *what* is happening. Guardrails and qualitative data tell you *why*.

In [ ]:
print('\n=== Retention Analysis Complete ===')
print(f'All outputs saved to: ../data/outputs/nb05/')
print(f'Plots:')
print('  - nb05_01_nweek_retention_curve.png')
print('  - nb05_02_classic_vs_rolling.png')
print('  - nb05_03_kaplan_meier.png')
print('  - nb05_03_km_by_plan.png')
print('  - nb05_04_curve_fitting.png')
print('  - nb05_05_churn_risk_features.png')